# Table 3 — Baseline OLS and IV Estimates
Replicates Table 3 of Friedberg & Isaac (2024).

Outcome: `married` (1 if same-sex married couple, 0 if cohabiting)

Six columns:
- Col 1: OLS, no income controls
- Col 2: IV,  no income controls
- Col 3: OLS, + earnings split
- Col 4: IV,  + earnings split
- Col 5: OLS, + 5th-order earnings poly + control function
- Col 6: IV,  + 5th-order earnings poly + control function

All specs include year FE, state FE, and the baseline covariate set.
IV instrument: predicted marriage subsidy (from Lasso-predicted earnings).

In [1]:
# Cell 1 — Imports and load data
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.linalg as la
import warnings
warnings.filterwarnings('ignore')

from linearmodels.iv import IV2SLS
from statsmodels.tools import add_constant as sm_add_const

IN_PKL = "/Users/trinityrose/Desktop/acs_ssc_msub_v2.pkl"

df = pd.read_pickle(IN_PKL)
print(f"Loaded: {df.shape}")
print(f"Married share: {df['sscouple_mar'].mean():.3f}")

Loaded: (38279, 123)
Married share: 0.422


In [2]:
# Cell 2 — Prepare variables
df["married"]      = df["sscouple_mar"].astype(float)
df["msub_obs_k"]   = df["msub_total_obs"] / 1000
df["msub_hat_k"]   = df["msub_total_hat"] / 1000
df["legal_marriage"] = df["staterecog_policy"].astype(float)
df["male"]         = df["r_male"].astype(float)
df["any_children"] = df["c_anychildren"].astype(float)
df["n_children"]   = df["c_children"].astype(float)
df["age_older"]    = df["c_agemax"].astype(float)
df["age_diff"]     = df["c_agediff"].astype(float)
df["edu_max"]      = df["c_edumax"].astype(float)
df["edu_diff"]     = df["c_edudiff"].astype(float)
df["same_race"]    = df["c_racecomp"].astype(float)
df["medicaid_exp"] = df["medicaid_exp"].astype(float)
df["earn_split_obs"] = df["c_incearn_split"].astype(float)
df["earn_split_hat"] = df["hat_c_split"].astype(float)

for i in range(1, 6):
    df[f"hat_earn{i}"]  = df[f"hat_c_incearn{i}"].astype(float)
    df[f"hat_split{i}"] = df[f"hat_c_split{i}"].astype(float)

df["year_fe"]  = df["year"].astype(str)
df["state_fe"] = df["statefip"].astype(str)

key_cols = ["married", "msub_obs_k", "msub_hat_k", "legal_marriage",
            "male", "any_children", "n_children", "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race", "medicaid_exp",
            "earn_split_obs", "earn_split_hat"]
df = df.dropna(subset=key_cols).copy()
print(f"Analysis sample: {len(df):,}")
print(f"Married share:   {df['married'].mean():.3f}  (paper: 0.432)")

Analysis sample: 38,279
Married share:   0.422  (paper: 0.432)


In [3]:
# Cell 3 — Helpers
def make_dummies(df, col):
    return pd.get_dummies(df[col], prefix=col, drop_first=True, dtype=float)

def drop_collinear(X, tol=1e-8):
    """Remove columns causing rank deficiency via QR with column pivoting."""
    arr = X.values.astype(float)
    Q, R, P = la.qr(arr, pivoting=True)
    diag   = np.abs(np.diag(R))
    rank   = int(np.sum(diag > tol * diag[0]))
    keep   = sorted(P[:rank])
    return X.iloc[:, keep]

def run_spec(df, extra_controls=None, iv=False):
    extra_controls = extra_controls or []
    base = ["legal_marriage", "medicaid_exp", "male",
            "any_children", "n_children",
            "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race"]
    year_dums  = make_dummies(df, "year_fe")
    state_dums = make_dummies(df, "state_fe")
    X_cols = base + extra_controls
    X = pd.concat([df[X_cols], year_dums, state_dums], axis=1).astype(float)
    X = sm_add_const(X)
    y = df["married"]
    if not iv:
        X.insert(1, "msub_obs_k", df["msub_obs_k"].values)
        X = drop_collinear(X)
        res = IV2SLS(y, X, None, None).fit(cov_type="robust")
    else:
        endog = df[["msub_obs_k"]]
        instr = df[["msub_hat_k"]]
        X = drop_collinear(X)
        res = IV2SLS(y, X, endog, instr).fit(cov_type="robust")
    return res

def first_stage_info(res):
    try:
        fs    = res.first_stage
        indiv = fs.individual["msub_obs_k"]
        c     = float(indiv.params["msub_hat_k"])
        se    = float(indiv.std_errors["msub_hat_k"])
        p     = float(indiv.pvalues["msub_hat_k"])
        f     = float(fs.diagnostics.loc["f.stat", "msub_obs_k"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

print("Helpers defined.")

Helpers defined.


In [4]:
# Cell 4 — Run all 6 columns
poly_cols = ([f"hat_earn{i}" for i in range(1, 6)] +
             [f"hat_split{i}" for i in range(1, 6)])

print("Col 1: OLS no income controls...")
col1 = run_spec(df, iv=False)

print("Col 2: IV  no income controls...")
col2 = run_spec(df, iv=True)

print("Col 3: OLS + earnings split...")
col3 = run_spec(df, extra_controls=["earn_split_obs"], iv=False)

print("Col 4: IV  + earnings split...")
col4 = run_spec(df, extra_controls=["earn_split_hat"], iv=True)

print("Col 5: OLS + poly + control function...")
col5 = run_spec(df, extra_controls=poly_cols, iv=False)

print("Col 6: IV  + poly + control function...")
col6 = run_spec(df, extra_controls=poly_cols, iv=True)

print("All columns estimated.")

Col 1: OLS no income controls...
Col 2: IV  no income controls...
Col 3: OLS + earnings split...
Col 4: IV  + earnings split...
Col 5: OLS + poly + control function...
Col 6: IV  + poly + control function...
All columns estimated.


In [5]:
# Cell 5 — Display table
def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return "   "

def get_coef_se_p(res, name):
    if name not in res.params.index:
        return None, None, None
    return res.params[name], res.std_errors[name], res.pvalues[name]

def first_stage_info(res):
    try:
        fs  = res.first_stage.results["msub_obs_k"]
        c   = fs.params["msub_hat_k"]
        se  = fs.std_errors["msub_hat_k"]
        p   = fs.pvalues["msub_hat_k"]
        f   = res.first_stage.diagnostics["f.stat"].values[0]
        return c, se, p, f
    except Exception:
        return np.nan, np.nan, np.nan, np.nan

results  = [col1, col2, col3, col4, col5, col6]
is_iv    = [False, True, False, True, False, True]
mean_dv  = df["married"].mean()

sep = "-" * 105

print("=" * 105)
print("TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)")
print("=" * 105)
hdr = f"{'':42}" + "".join(f"{'OLS' if not v else 'IV':>11}" for v in is_iv)
print(hdr)
print(sep)

def print_row(label, varname, paper_vals):
    # Coefficient row
    line = f"{label:42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if c is not None:
            line += f"  {c:7.3f}{stars(p)}"
        else:
            line += f"  {'—':>10}"
    print(line)
    # SE row
    line = f"{'':42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if se is not None:
            line += f"  ({se:6.3f})  "
        else:
            line += f"  {'':>10}"
    print(line)
    # Paper row
    print(f"  {'(paper)':40}" + "".join(f"  {v:>10}" for v in paper_vals))
    print()

print_row("Marriage subsidy ($1,000s)", "msub_obs_k",
          ["0.005***", "0.008***", "0.004***", "0.009*  ", "0.005***", "0.014***"])
print_row("Legal marriage", "legal_marriage",
          ["0.066***", "0.066***", "0.066***", "0.066***", "0.116***", "0.116***"])
print_row("State Medicaid expansion", "medicaid_exp",
          ["0.010   ", "0.010   ", "0.009   ", "0.010   ", "0.035***", "0.034***"])

print(sep)

# First stage
print(f"{'First stage coef':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  {c:7.3f}{stars(p)}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()

print(f"{'':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  ({se:6.3f})  ", end="")
    else:
        print(f"  {'':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'0.463***':>10}  {'—':>10}  {'0.408***':>10}  {'—':>10}  {'0.420***':>10}")
print()

print(f"{'First stage F-stat':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        _, _, _, f = first_stage_info(res)
        print(f"  {f:>10.1f}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'[474.7]':>10}  {'—':>10}  {'[221.0]':>10}  {'—':>10}  {'[261.3]':>10}")

print(sep)
print(f"{'Mean dep var':42}" + "".join(f"  {mean_dv:>10.3f}" for _ in results))
print(f"{'N':42}" + "".join(f"  {int(res.nobs):>10,}" for res in results))
print(f"  {'(paper N)':40}" + "".join(f"  {'37,234':>10}" for _ in results))
print("=" * 105)

TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)
                                                  OLS         IV        OLS         IV        OLS         IV
---------------------------------------------------------------------------------------------------------
Marriage subsidy ($1,000s)                    0.009***    0.015***    0.007***    0.029***    0.009***    0.020***
                                            ( 0.001)    ( 0.005)    ( 0.001)    ( 0.006)    ( 0.001)    ( 0.008)  
  (paper)                                     0.005***    0.008***    0.004***    0.009*      0.005***    0.014***

Legal marriage                                0.062***    0.062***    0.063***    0.062***    0.062***    0.061***
                                            ( 0.010)    ( 0.010)    ( 0.010)    ( 0.010)    ( 0.010)    ( 0.010)  
  (paper)                                     0.066***    0.066***    0.066***    0.066***    0.116***    0.116***

State Medicaid expansion        

In [6]:
# Cell 6 — Elasticity
c6, _, _ = get_coef_se_p(col6, "msub_obs_k")
mean_msub = df["msub_obs_k"].mean()
elasticity = c6 * (mean_msub / mean_dv)
print(f"IV coef (col 6):     {c6:.4f}")
print(f"Mean subsidy ($k):   {mean_msub:.3f}")
print(f"Mean married:        {mean_dv:.3f}")
print(f"Implied elasticity:  {elasticity:.4f}  (paper: 0.011)")

IV coef (col 6):     0.0200
Mean subsidy ($k):   0.688
Mean married:        0.422
Implied elasticity:  0.0326  (paper: 0.011)


In [7]:
# Debug cell — run this, then we'll fix first_stage_info
fs = col2.first_stage
print(type(fs))
print(dir(fs))
print()
print("params type:", type(fs.params))
print("params:\n", fs.params)
print()
print("diagnostics type:", type(fs.diagnostics))
print("diagnostics:\n", fs.diagnostics)

<class 'linearmodels.iv.results.FirstStageResults'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_cov_config', '_cov_type', '_reg', '_repr_html_', 'dep', 'diagnostics', 'endog', 'exog', 'individual', 'instr', 'summary', 'weights']



AttributeError: 'FirstStageResults' object has no attribute 'params'

In [ ]:
fs = col2.first_stage
print([x for x in dir(fs) if not x.startswith('_')])

['dep', 'diagnostics', 'endog', 'exog', 'individual', 'instr', 'summary', 'weights']
